In [2]:
import pandas as pd
import numpy as np
from scipy import stats
import itertools

In [3]:
df = pd.read_csv("gpt_output.csv")
df["HCV_active"] = df["HCV"].isin(["acute", "chronic"]).astype(int)

In [4]:
measures = [
    "Withdrawal", "Public_prop", "Recept_sharing", "Partners",
    "Drug_out", "Partner_HCV_prop", "Treatment",
    "Injections", "Alt_route", "HCV_active"
]

In [5]:
def paired_bootstrap(df, outcome_col, agent_col="Agent", condition_col="Homeless", n_boot=1000, seed=42):
    np.random.seed(seed)

    wide = df.pivot(index=agent_col,
                    columns=condition_col,
                    values=outcome_col).dropna()

    agents = wide.index.values
    diffs = []

    for _ in range(n_boot):
        sampled_agents = np.random.choice(agents, size=len(agents), replace=True)
        sample = wide.loc[sampled_agents]
        diffs.append((sample[1] - sample[0]).mean())

    diffs = np.array(diffs)
    point_est = (wide[1] - wide[0]).mean() #original (non-bootstrap) estimate

    return {
        "estimate": point_est,
        "ci_lower": np.percentile(diffs, 2.5),
        "ci_upper": np.percentile(diffs, 97.5),
        "bootstrap_sd": diffs.std(ddof=1)
    }

In [17]:
results = []

for v in measures:
    result = paired_bootstrap(df, outcome_col=v)

    agg = df.groupby(["Agent", "Homeless"])[v].mean().reset_index()
    wide = agg.pivot(index="Agent", columns="Homeless", values=v)
    wide.columns = ["stable", "homeless"]
    wide = wide.dropna()
    diff = wide["homeless"] - wide["stable"]

    d = diff.mean() / (diff.std(ddof=1) + 1e-8)

    results.append({
        "Variable": v,
        #"Difference": result["estimate"],
        "CI_lower": result["ci_lower"],
        "CI_upper": result["ci_upper"],
        "Cohens_d": d
    })

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by="Cohens_d", ascending=False)

print(results_df)

           Variable  CI_lower  CI_upper  Cohens_d
1       Public_prop  0.498616  0.531742  3.941642
9        HCV_active  0.022321  0.075893  0.226744
3          Partners  0.012054  0.198672  0.146240
2    Recept_sharing  0.061259  1.071507  0.128899
4          Drug_out -0.478884  0.225246 -0.045140
5  Partner_HCV_prop -0.001786 -0.000223 -0.134538
0        Withdrawal -0.345982 -0.031194 -0.152516
7        Injections -1.370592 -0.421763 -0.237528
8         Alt_route -0.308549 -0.095949 -0.257770
6         Treatment -0.160714 -0.066964 -0.311402


In [7]:
results_df.to_csv("outputs/claude_results_table.csv", index=False, encoding="utf-8")

In [8]:
table = pd.crosstab(df["Neighborhood"], df["Homeless"])
print(pd.crosstab(df["Neighborhood"], df["Homeless"], normalize="columns")) 

Homeless             0         1
Neighborhood                    
central       0.642857  0.598214
north side    0.017857  0.000000
south side    0.125000  0.160714
west side     0.214286  0.241071


In [9]:
table = pd.crosstab(df["Race"], df["Neighborhood"], normalize="index")
print(table)

Neighborhood   central  north side  south side  west side
Race                                                     
Hispanic      0.526786    0.000000    0.071429   0.401786
NHBlack       0.437500    0.000000    0.375000   0.187500
NHWhite       0.705357    0.026786    0.053571   0.214286
Other         0.812500    0.008929    0.071429   0.107143


In [10]:
def safe_corr(x, y):
    df_xy = pd.concat([x, y], axis=1).dropna()

    x_clean = df_xy.iloc[:, 0]
    y_clean = df_xy.iloc[:, 1]

    if x_clean.nunique() < 2 or y_clean.nunique() < 2:
        return np.nan
    return x_clean.corr(y_clean)


results = []
for a, b in itertools.combinations(measures, 2):

    corr = safe_corr(df[a], df[b])
    results.append({
        "var1": a,
        "var2": b,
        "correlation": corr
    })

corr_df = pd.DataFrame(results)
threshold = 0.3

strong_corr = corr_df[(corr_df["correlation"].abs() >= threshold)
    ].sort_values(by="correlation", key=lambda s: s.abs(), ascending=False)

strong_corr = strong_corr.reset_index(drop=True)

print(strong_corr)

                var1            var2  correlation
0   Partner_HCV_prop      HCV_active     0.883611
1         Withdrawal       Alt_route     0.742814
2         Withdrawal      Injections     0.708098
3     Recept_sharing        Drug_out     0.644022
4         Withdrawal  Recept_sharing     0.543709
5     Recept_sharing      Injections     0.510143
6         Injections       Alt_route     0.501685
7           Partners        Drug_out     0.494023
8     Recept_sharing        Partners     0.396004
9         Withdrawal        Drug_out     0.387234
10        Withdrawal        Partners     0.386583
11    Recept_sharing       Alt_route     0.373720
12          Drug_out      Injections     0.346876
13          Partners       Alt_route     0.334485
14          Drug_out       Alt_route     0.307802
15          Partners      Injections     0.302594


In [11]:
#cohens d function
def cohens_d(x0, x1):
    x0 = x0.dropna()
    x1 = x1.dropna()

    if len(x0) < 2 or len(x1) < 2:
        return np.nan

    n0, n1 = len(x0), len(x1)
    s0, s1 = x0.std(ddof=1), x1.std(ddof=1)

    pooled = np.sqrt(((n0 - 1)*s0**2 + (n1 - 1)*s1**2) / (n0 + n1 - 2))

    if pooled == 0 or np.isnan(pooled):
        return np.nan

    return (x1.mean() - x0.mean()) / pooled


In [12]:
baseline_vars = {
    "prior_injection": "Prev_Injection",
    "prior_sharing": "Prev_recept_sharing",
    "prior_network": "Prev_friend_entry",
}
output_vars = {
    "prior_injection": "Injections",
    "prior_sharing": "Recept_sharing",
    "prior_network": "Partner_HCV_prop",
}

results = []

for key in baseline_vars:
    base = baseline_vars[key]
    out = output_vars[key]

    sub = df[[base, out, "Homeless"]].dropna().copy()
    sub["change"] = sub[out] - sub[base]

    stable = sub[sub["Homeless"] == 0]["change"]
    homeless = sub[sub["Homeless"] == 1]["change"]

    d = cohens_d(stable, homeless)

    results.append({
        "relationship": f"{base} → {out}",
        "std_mean_change_effect": d
    })

result = pd.DataFrame(results)
print(result)

                           relationship  std_mean_change_effect
0           Prev_Injection → Injections               -0.271767
1  Prev_recept_sharing → Recept_sharing                0.065461
2  Prev_friend_entry → Partner_HCV_prop               -0.186214


In [13]:
df = df.copy()
df["experience"] = df["Age"] - df["Age_Started"]
df["experience_group"] = np.where(df["experience"] >= 3, "Experienced", "Not experienced")
results = []

for y in measures:
    exp0 = df[df["experience_group"] == "Not experienced"]
    exp1 = df[df["experience_group"] == "Experienced"]

    d_not_exp = cohens_d(
        exp0[exp0["Homeless"] == 0][y],
        exp0[exp0["Homeless"] == 1][y]
    )
    d_exp = cohens_d(
        exp1[exp1["Homeless"] == 0][y],
        exp1[exp1["Homeless"] == 1][y]
    )
    results.append({
        "Outcome": y,
        "Difference (Experienced - Not)": d_exp - d_not_exp
    })

experience_results = pd.DataFrame(results).round(4)
print(experience_results)

            Outcome  Difference (Experienced - Not)
0        Withdrawal                          0.0097
1       Public_prop                          0.5137
2    Recept_sharing                          0.0160
3          Partners                          0.0066
4          Drug_out                         -0.0901
5  Partner_HCV_prop                         -0.0015
6         Treatment                          0.0865
7        Injections                          0.0279
8         Alt_route                          0.0849
9        HCV_active                          0.0448


In [14]:
df["age_group"] = np.where(df["Age"] >= df["Age"].median(), "Old", "Young")
results = []

for y in measures:
    young_stable = df[(df["age_group"] == "Young") & (df["Homeless"] == 0)][y].dropna()
    young_homeless = df[(df["age_group"] == "Young") & (df["Homeless"] == 1)][y].dropna()

    old_stable = df[(df["age_group"] == "Old") & (df["Homeless"] == 0)][y]
    old_homeless = df[(df["age_group"] == "Old") & (df["Homeless"] == 1)][y]

    young_d = cohens_d(young_stable, young_homeless)
    old_d = cohens_d(old_stable, old_homeless)

    results.append({
        "Outcome": y,
        "Difference (Old - Young)": old_d - young_d if pd.notna(old_d) and pd.notna(young_d) else np.nan
    })

age_results = pd.DataFrame(results).round(4)
print(age_results)

            Outcome  Difference (Old - Young)
0        Withdrawal                   -0.1013
1       Public_prop                    1.3784
2    Recept_sharing                    0.0801
3          Partners                   -0.0642
4          Drug_out                    0.0283
5  Partner_HCV_prop                   -0.0011
6         Treatment                   -0.0530
7        Injections                   -0.0127
8         Alt_route                   -0.1091
9        HCV_active                   -0.1789


In [15]:
results = []
for y in measures:
    sub = df[["Gender", "Homeless", y]].dropna()

    female_stable = sub[(sub["Gender"] == "Female") & (sub["Homeless"] == 0)][y]
    female_homeless = sub[(sub["Gender"] == "Female") & (sub["Homeless"] == 1)][y]

    male_stable = sub[(sub["Gender"] == "Male") & (sub["Homeless"] == 0)][y]
    male_homeless = sub[(sub["Gender"] == "Male") & (sub["Homeless"] == 1)][y]

    female_d = cohens_d(female_stable, female_homeless)
    male_d = cohens_d(male_stable, male_homeless)

    results.append({
        "Outcome": y,
        "Difference (Female - Male)": female_d - male_d if pd.notna(female_d) and pd.notna(male_d) else np.nan
    })

gender_results = pd.DataFrame(results).round(4)
print(gender_results)

            Outcome  Difference (Female - Male)
0        Withdrawal                      0.0214
1       Public_prop                      0.1433
2    Recept_sharing                     -0.0806
3          Partners                     -0.0866
4          Drug_out                      0.0367
5  Partner_HCV_prop                      0.0032
6         Treatment                      0.0238
7        Injections                      0.0388
8         Alt_route                      0.0147
9        HCV_active                      0.0834


In [16]:
results = []

for y in measures:
    for race, g in df.groupby("Race"):
        stable = g[g["Homeless"] == 0][y]
        homeless = g[g["Homeless"] == 1][y]
        d = cohens_d(stable, homeless)

        results.append({
            "Outcome": y,
            "Race": race,
            "Cohens_d": d
        })
race_d_df = pd.DataFrame(results)

race_summary = race_d_df.groupby("Outcome")["Cohens_d"].agg(
    race_range=lambda x: x.max() - x.min(),
    mean="mean"
).reset_index()

print(race_summary)

            Outcome  race_range      mean
0         Alt_route    0.405746 -0.274803
1          Drug_out    0.080048 -0.035778
2        HCV_active    0.170367  0.164366
3        Injections    0.098039 -0.052504
4  Partner_HCV_prop    0.007235 -0.003334
5          Partners    0.195146  0.094740
6       Public_prop    3.344802  6.094747
7    Recept_sharing    0.105106  0.055671
8         Treatment    0.222383 -0.437960
9        Withdrawal    0.300994 -0.132876


In [21]:
results = []

for y in measures:
    for race, g in df.groupby("Race"):
        stable = g[g["Homeless"] == 0][y]
        homeless = g[g["Homeless"] == 1][y]
        d = cohens_d(stable, homeless)

        results.append({
            "Outcome": y,
            "Race": race,
            "Cohens_d": d
        })
race_d_df = pd.DataFrame(results)

In [20]:
extremes = race_d_df.loc[
    race_d_df.groupby("Outcome")["Cohens_d"].idxmax()
].rename(columns={"Race": "max_race", "Cohens_d": "max_d"})

mins = race_d_df.loc[
    race_d_df.groupby("Outcome")["Cohens_d"].idxmin()
].rename(columns={"Race": "min_race", "Cohens_d": "min_d"})

extreme_summary = extremes.merge(
    mins[["Outcome", "min_race", "min_d"]],
    on="Outcome"
)

print(extreme_summary)

            Outcome  max_race     max_d  min_race     min_d
0         Alt_route  Hispanic -0.091627   NHBlack -0.497372
1          Drug_out   NHBlack  0.007716   NHWhite -0.072333
2        HCV_active     Other  0.277544  Hispanic  0.107178
3        Injections     Other  0.006303   NHBlack -0.091736
4  Partner_HCV_prop     Other  0.000000   NHWhite -0.007235
5          Partners  Hispanic  0.225521     Other  0.030375
6       Public_prop   NHBlack  7.762406   NHWhite  4.417604
7    Recept_sharing     Other  0.116928   NHBlack  0.011822
8         Treatment   NHWhite -0.333446  Hispanic -0.555829
9        Withdrawal     Other -0.014841   NHBlack -0.315835
